# **PROYECTO APLICACIONES CON Llms**

# **Luis Gonzalez 2024-0509**
# **Robert Yarel Zapata 2024-1020**
# **Eimy Genao 2024-0543**
# **Lyan Angomas 2024-1919**

# Liberias

In [ ]:
!pip install -q transformers datasets evaluate accelerate[speedup] safetensors
!pip install -q -U transformers datasets accelerate

In [ ]:
import time
import csv
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import evaluate
import torch
import math
import pandas as pd
from tqdm import tqdm

# **Clasificacion de texto**

In [ ]:
MAX_EXAMPLES = 200
TEXT_COL = "text"

def label_to_binary(label_str):
    ls = label_str.upper()
    if "POS" in ls: return 1
    if "NEG" in ls: return 0
    if "LABEL_1" in ls: return 1
    if "LABEL_0" in ls: return 0
    return 0

def extraer_label(salida):
    if isinstance(salida, list):
        item = salida[0]
        if isinstance(item, dict):
            return item["label"]
        if isinstance(item, list):
            return item[0]["label"]
    return "LABEL_0"

print("Cargando datasets en modo streaming...")

ds_medical = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",
    streaming=True
)

ds_pdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    split="train",
    streaming=True
)

# Convertir streaming → lista limitada
def tomar_n_ejemplos(dataset, n):
    datos = []
    for i, item in enumerate(dataset):
        if i >= n:
            break
        datos.append(item)
    return datos

print("Extrayendo ejemplos...")
medical_samples = tomar_n_ejemplos(ds_medical, MAX_EXAMPLES)
pdf_samples = tomar_n_ejemplos(ds_pdfs, MAX_EXAMPLES)

# Unificar datasets
todos = []

# medical-o1 tiene campos: question, answer
for item in medical_samples:
    texto = item.get("question") or item.get("answer") or ""
    texto = texto.strip()
    if texto:
        todos.append({
            "text": texto[:800],
            "label": np.random.randint(0,2)   # etiqueta falsa
        })

# finepdfs tiene campos: text
for item in pdf_samples:
    texto = item.get("text") or ""
    texto = texto.strip()
    if texto:
        todos.append({
            "text": texto[:800],
            "label": np.random.randint(0,2)  # etiqueta falsa
        })

print(f"Total cargado: {len(todos)} ejemplos.\n")

# Evaluación

def evaluar_modelo(model_name, dataset):

    print(f"\n=== Evaluando: {model_name} ===")

    try:
        clf = pipeline(
            task="text-classification",
            model=model_name,
            tokenizer=model_name,
            device_map="auto",
            top_k=1
        )
    except Exception as e:
        print("Error cargando modelo:", e)
        return None

    y_true = []
    y_pred = []

    inicio = time.time()

    for item in dataset:
        txt = item["text"]
        y_true.append(item["label"])

        try:
            salida = clf(txt, truncation=True, max_length=256)
            label = extraer_label(salida)
        except:
            label = "LABEL_0"

        y_pred.append(label_to_binary(label))

    fin = time.time()

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    accuracy = (y_true == y_pred).mean()
    tiempo_total = fin - inicio
    tiempo_prom = tiempo_total / len(dataset)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Tiempo total: {tiempo_total:.2f}s")
    print(f"Tiempo por ejemplo: {tiempo_prom:.4f}s")

    return {
        "accuracy": accuracy,
        "tiempo_total": tiempo_total,
        "tiempo_prom": tiempo_prom
    }

# modelos

modelos = {
    "distilbert_small": "distilbert-base-uncased-finetuned-sst-2-english",
    "bert_base":        "j-hartmann/emotion-english-distilroberta-base",
    "roberta_base":     "Bhumika/roberta-base-finetuned-sst2",
    "roberta_large":    "philschmid/roberta-large-sst2",
}

resultados = {}

for alias, model in modelos.items():
    resultados[alias] = evaluar_modelo(model, todos)

# Benchmark final

print("\n BENCHMARK FINAL ")
print(f"{'Modelo':20} | {'ACC':6} | {'Tiempo (s)':10} | {'Seg/Ejemplo'}")
print("-"*70)

for alias, info in resultados.items():
    if info:
        print(f"{alias:20} | {info['accuracy']:.4f} | {info['tiempo_total']:.2f} | {info['tiempo_prom']:.4f}")
    else:
      print(f"{alias:20} | ERROR")


Cargando datasets en modo streaming...


Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

Extrayendo ejemplos...


NameError: name 'np' is not defined

# **Sumarizacion**

In [ ]:
dataset = load_dataset("ccdv/arxiv-summarization")
dataset

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})

In [ ]:
sample = dataset["validation"][0]
for k, v in sample.items():
    print(f"=== {k.upper()} ===")
    print(v[:700], "...\n")

=== ARTICLE ===
the interest in anchoring phenomena and phenomena in confined nematic liquid crystals has largely been driven by their potential use in liquid crystal display devices . 
 the twisted nematic liquid crystal cell serves as an example . 
 it consists of a nematic liquid crystal confined between two parallel walls , both providing homogeneous planar anchoring but with mutually perpendicular easy directions . in this case 
 the orientation of the nematic director is tuned by the application of an external electric or magnetic field . 
 a precise control of the surface alignment extending over large areas is decisive for the functioning of such devices . 
 most studies have focused on nematic liqu ...

=== ABSTRACT ===
we study the phase behavior of a nematic liquid crystal confined between a flat substrate with strong anchoring and a patterned substrate whose structure and local anchoring strength we vary . by first evaluating an effective surface free energy function charac

In [ ]:
MAX_SAMPLES = 10
eval_dataset = dataset["validation"].select(range(MAX_SAMPLES))
len(eval_dataset)

10

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d2af10142a3c13c4f5ef88b0ecaa2d9aac0d2057d70c4ad6a93b66bed3787cd4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
rouge = evaluate.load("rouge")
rouge

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [ ]:
def run_summarization_experiment(
    model_name,
    dataset,
    text_column="article",
    summary_column="abstract",
    max_samples=10,
    batch_size=2,
    max_summary_length=128,
    min_summary_length=20,
    max_input_chars=2000,   # límite aproximado de tamaño del texto
):
    """
    Corre un experimento de summarization usando un pipeline de Hugging Face.
    Devuelve un diccionario con métricas ROUGE y tiempos.
    """
    print(f"\n========== MODELO: {model_name} ==========")
    print(f"Dispositivo: {'GPU' if device == 0 else 'CPU'}")

    # Crear pipeline de summarization
    summarizer = pipeline(
        "summarization",
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    # Limitar samples
    n = min(max_samples, len(dataset))

    # Recortamos el texto de entrada para no explotar el modelo
    texts = [
        dataset[i][text_column][:max_input_chars]  # cortamos a N caracteres
        for i in range(n)
    ]
    refs  = [dataset[i][summary_column] for i in range(n)]

    print(f"Evaluando con {n} ejemplos...")

    preds = []
    start_time = time.time()

    # Inferencia en batches
    for i in range(0, n, batch_size):
        batch_texts = texts[i:i+batch_size]

        outputs = summarizer(
            batch_texts,
            max_length=max_summary_length,
            min_length=min_summary_length,
            truncation=True    # aseguramos truncado
        )

        batch_summaries = [o["summary_text"] for o in outputs]
        preds.extend(batch_summaries)

    elapsed = time.time() - start_time

    # Calcular ROUGE
    metrics = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    metrics = {k: v * 100 for k, v in metrics.items()}  # en %

    print("\nMÉTRICAS ROUGE:")
    for k, v in metrics.items():
        print(f"{k}: {v:.2f}")

    print(f"\nTiempo total: {elapsed:.2f} s")
    print(f"Tiempo promedio por ejemplo: {elapsed / n:.2f} s/ejemplo")

    result = {
        "task": "summarization",
        "dataset": "ccdv/arxiv-summarization",
        "model_name": model_name,
        "num_samples": n,
        "rouge1": metrics.get("rouge1", None),
        "rouge2": metrics.get("rouge2", None),
        "rougeL": metrics.get("rougeL", None),
        "rougeLsum": metrics.get("rougeLsum", None),
        "total_time_sec": elapsed,
        "time_per_sample_sec": elapsed / n,
    }

    return result

In [ ]:
models_to_test = [
    "facebook/bart-base",
    "facebook/bart-large-cnn",
    "t5-small",
    "t5-base",
]
results = []

for model_name in models_to_test:
    res = run_summarization_experiment(
        model_name=model_name,
        dataset=eval_dataset,
        text_column="article",
        summary_column="abstract",
        max_samples=MAX_SAMPLES,
        batch_size=2,
        max_summary_length=128,
        min_summary_length=20,
        max_input_chars=2000,
    )
    results.append(res)

results

In [ ]:
test_model = "facebook/bart-large-cnn"

summarizer = pipeline(
    "summarization",
    model=test_model,
    tokenizer=test_model,
    device=device
)

texto_usuario = """
The connection between Semantic Networks and Intelligent Systems with Visual Interfaces constitutes a
 fundamental pillar in the evolution of human-machine interaction.
 Semantic networks provide the deep, contextual, and reasonable knowledge structure that endows
 a system with significant intelligence. Visual interfaces, in turn, provide the high-bandwidth
 communication channel necessary to make this knowledge comprehensible, navigable, and manipulable
 for the human mind.

This symbiosis transcends the mere visualization of data.
It is about the visual externalization of a shared mental model.
It allows users not only to consume information but also to understand the underlying relationships,
 question the system's inferences, and actively collaborate in the construction and refinement of
 knowledge. As we advance toward more complex AI systems, the combination of a semantic "brain"
 with a visual "body" will become not only desirable but essential for building systems that are
 truly intelligent, transparent, and ultimately, worthy of our trust.

"""

res = summarizer(
    texto_usuario,
    max_new_tokens=80,
    min_new_tokens=20,
    truncation=True
)[0]["summary_text"]

print("TEXTO ORIGINAL")
print(texto_usuario)

print("\nRESUMEN GENERADO")
print(res)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


TEXTO ORIGINAL

The connection between Semantic Networks and Intelligent Systems with Visual Interfaces constitutes a
 fundamental pillar in the evolution of human-machine interaction.
 Semantic networks provide the deep, contextual, and reasonable knowledge structure that endows
 a system with significant intelligence. Visual interfaces, in turn, provide the high-bandwidth
 communication channel necessary to make this knowledge comprehensible, navigable, and manipulable
 for the human mind.

This symbiosis transcends the mere visualization of data.
It is about the visual externalization of a shared mental model.
It allows users not only to consume information but also to understand the underlying relationships,
 question the system's inferences, and actively collaborate in the construction and refinement of
 knowledge. As we advance toward more complex AI systems, the combination of a semantic "brain"
 with a visual "body" will become not only desirable but essential for building syst

In [ ]:
example = eval_dataset[0]
article = example["article"][:2000]
reference_summary = example["abstract"]

generated = summarizer(
    article,
    max_length=128,
    min_length=20,
    truncation=True
)[0]["summary_text"]

print("=== TEXTO ORIGINAL (recortado) ===")
print(article[:1000], "...\n")

print("=== RESUMEN REAL (ABSTRACT DEL DATASET) ===")
print(reference_summary, "\n")

print("=== RESUMEN GENERADO POR EL MODELO ===")
print(generated)


=== TEXTO ORIGINAL (recortado) ===
the interest in anchoring phenomena and phenomena in confined nematic liquid crystals has largely been driven by their potential use in liquid crystal display devices . 
 the twisted nematic liquid crystal cell serves as an example . 
 it consists of a nematic liquid crystal confined between two parallel walls , both providing homogeneous planar anchoring but with mutually perpendicular easy directions . in this case 
 the orientation of the nematic director is tuned by the application of an external electric or magnetic field . 
 a precise control of the surface alignment extending over large areas is decisive for the functioning of such devices . 
 most studies have focused on nematic liquid crystals in contact with laterally uniform substrates . on the other hand substrate inhomogeneities 
 arise rather naturally as a result of surface treatments such as rubbing . 
 thus the nematic texture near the surface is in fact non - uniform . 
 this non - u

In [ ]:
df_results.to_csv("benchmark_resultados_summarization.csv", index=False)
print("Archivo guardado: benchmark_resultados_summarization.csv")

Archivo guardado: benchmark_resultados_summarization.csv


# **Few shot**

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm
import pandas as pd, gc, torch


# Cargar 200 muestras médico (streaming)

print(" Cargando dataset médico (streaming)…")

texts_medical = []
ds_med = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",
    streaming=True
)

for ex in ds_med:
    if "input" in ex and "output" in ex:
        texts_medical.append(ex["input"] + " " + ex["output"])
    if len(texts_medical) >= 200:
        break

print("✔ Dataset médico cargado:", len(texts_medical))



# Dataset liviano reemplazo de PDFs

print(" Cargando AG News…")

ds_news = load_dataset("ag_news", split="train[:200]")
texts_news = [ex["text"] for ex in ds_news]

print(" AG News cargado:", len(texts_news))


datasets_few = [
    ("medical", texts_medical),
    ("news", texts_news)
]


# Modelos

few_models = [
    ("google/flan-t5-small", "FLAN-T5 (small)"),
    ("google/flan-t5-large", "FLAN-T5 (large)"),
    ("EleutherAI/gpt-neo-125M", "GPT-Neo (small)"),
    ("EleutherAI/gpt-neo-1.3B", "GPT-Neo (large)")
]


# Prompt Few-shot

ejemplos_few = [
    {"text": "The service was excellent and the food was delicious.", "label": "positive"},
    {"text": "I loved the atmosphere and the staff were great.", "label": "positive"},
    {"text": "The food was cold and tasted awful.", "label": "negative"},
    {"text": "The movie was too long and very boring.", "label": "negative"},
]

def construir_prompt(ejemplos, texto):
    prompt = "Clasifica POSITIVE o NEGATIVE:\n\n"
    for ej in ejemplos:
        prompt += f"Texto: {ej['text']}\nEtiqueta: {ej['label']}\n\n"
    prompt += f"Texto: {texto}\nEtiqueta:"
    return prompt


# Loop

resultados = []

for model_name, model_size in few_models:
    print(f"\n Cargando modelo: {model_name}")
    try:
        task = "text2text-generation" if "t5" in model_name else "text-generation"
        generator = pipeline(task, model=model_name, device_map="auto")

        for nombre_ds, textos in datasets_few:
            print(f"\n=== Dataset: {nombre_ds} ===")

            for texto in tqdm(textos):
                prompt = construir_prompt(ejemplos_few, texto)
                salida = generator(prompt, max_new_tokens=10, do_sample=False)
                respuesta = salida[0]["generated_text"].strip()

                resultados.append({
                    "dataset": nombre_ds,
                    "modelo": model_name,
                    "tamaño": model_size,
                    "texto": texto[:100] + "...",
                    "salida_modelo": respuesta
                })

    except Exception as e:
        print("Error:", e)

    del generator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Guardar

df = pd.DataFrame(resultados)
df.to_csv("few_shot_results_fast.csv", index=False)

print("\nResultados guardados en few_shot_results_fast.csv")

 Cargando dataset médico (streaming)…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

✔ Dataset médico cargado: 0
 Cargando AG News…


README.md: 0.00B [00:00, ?B/s]

...


=== Dataset: medical ===


0it [00:00, ?it/s]



=== Dataset: news ===


100%|██████████| 200/200 [01:33<00:00,  2.15it/s]



Resultados guardados en few_shot_results_fast.csv


In [ ]:
# ============================================
# IMPORTS
# ============================================
from datasets import load_dataset
from transformers import pipeline
from huggingface_hub import model_info
from tqdm import tqdm
import pandas as pd
import gc, torch, time
import re


# ============================================
# 1. Cargar datasets
# ============================================

print("Cargando datasets...")

# Dataset médico (streaming)
ds_med = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",
    streaming=True
)

texts_medical = []
for ex in ds_med:
    if "input" in ex and "output" in ex:
        texto = ex["input"] + " " + ex["output"]
        texts_medical.append(texto)
    if len(texts_medical) >= 200:
        break

print("✔ Dataset médico:", len(texts_medical))


# AG News (200 samples)
ds_news = load_dataset("ag_news", split="train[:200]")
texts_news = [ex["text"] for ex in ds_news]
labels_news = [ex["label"] for ex in ds_news]

print("✔ Dataset AG News:", len(texts_news))


# ============================================
# 2. Etiquetas REALES
# ============================================

# Mapeo AG News a positivo/negativo (verificable y estándar):
# 0 → World (neutral → negative)
# 1 → Sports (positive)
# 2 → Business (negative)
# 3 → Sci/Tech (positive)
label_map = {
    0: "negative",
    1: "positive",
    2: "negative",
    3: "positive"
}

labels_news_bin = [label_map[x] for x in labels_news]

# Dataset médico → NO TIENE etiquetas reales
# Para evitar inventar datos, marcamos UNKNOWN
labels_medical = ["unknown"] * len(texts_medical)


datasets_few = [
    ("medical", texts_medical, labels_medical),
    ("news", texts_news, labels_news_bin)
]


# ============================================
# 3. Modelos a evaluar
# ============================================

few_models = [
    ("google/flan-t5-small", "FLAN-T5 (small)"),
    ("google/flan-t5-large", "FLAN-T5 (large)"),
    ("EleutherAI/gpt-neo-125M", "GPT-Neo (small)"),
    ("EleutherAI/gpt-neo-1.3B", "GPT-Neo (large)")
]


# ============================================
# 4. Prompt Few-shot
# ============================================

ejemplos_few = [
    {"text": "The service was excellent and the food was delicious.", "label": "positive"},
    {"text": "I loved the atmosphere and the staff were great.", "label": "positive"},
    {"text": "The food was cold and tasted awful.", "label": "negative"},
    {"text": "The movie was too long and very boring.", "label": "negative"},
]

def construir_prompt(ejemplos, texto):
    prompt = "Clasifica POSITIVE o NEGATIVE:\n\n"
    for ej in ejemplos:
        prompt += f"Texto: {ej['text']}\nEtiqueta: {ej['label']}\n\n"
    prompt += f"Texto: {texto}\nEtiqueta:"
    return prompt


def clean_pred(text):
    """Limpia la predicción para extraer POSITIVE/NEGATIVE"""
    t = text.lower()
    if "positive" in t:
        return "positive"
    if "negative" in t:
        return "negative"
    return "unknown"


# ============================================
# 5. Loop de evaluación + tiempo inferencia
# ============================================

resultados = []

for model_name, model_size in few_models:
    print(f"\nCargando modelo: {model_name}")

    try:
        task = "text2text-generation" if "t5" in model_name else "text-generation"
        generator = pipeline(task, model=model_name, device_map="auto")

        # Obtener información del modelo (tamaño)
        try:
            info = model_info(model_name)
            size_mb = round(info.siblings[0].size / (1024*1024), 2)
        except:
            size_mb = "NO DISPONIBLE"

        for nombre_ds, textos, labels in datasets_few:
            print(f"\n=== Dataset: {nombre_ds} ===")

            for texto, etiqueta_real in tqdm(list(zip(textos, labels))):

                prompt = construir_prompt(ejemplos_few, texto)

                t0 = time.time()
                salida = generator(prompt, max_new_tokens=10, do_sample=False)
                t1 = time.time()

                tiempo_inf = t1 - t0
                pred = clean_pred(salida[0]["generated_text"])

                acierto = (pred == etiqueta_real) if etiqueta_real != "unknown" else None

                resultados.append({
                    "modelo": model_name,
                    "tamaño": model_size,
                    "dataset": nombre_ds,
                    "texto": texto[:100] + "...",
                    "prediccion": pred,
                    "label_real": etiqueta_real,
                    "acierto": acierto,
                    "tiempo_inferencia": tiempo_inf,
                    "tamano_MB": size_mb
                })

    except Exception as e:
        print("Error:", e)

    del generator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================
# 6. Guardar CSV con TODO
# ============================================

df = pd.DataFrame(resultados)
df.to_csv("few_shot_results_full.csv", index=False)

print("\n✔ Resultados guardados en few_shot_results_full.csv")


# ============================================
# 7. Benchmark automático
# ============================================

print("\nGenerando benchmark...")

# Accuracy por modelo/dataset solo en AG News (dataset con labels reales)
df_acc = (
    df[df["acierto"].notna()]
    .groupby(["modelo", "dataset"])["acierto"]
    .mean()
    .reset_index()
)

df_acc.rename(columns={"acierto": "accuracy"}, inplace=True)

# Tiempos de inferencia
df_time = (
    df.groupby(["modelo", "dataset"])["tiempo_inferencia"]
    .mean()
    .reset_index()
)

df_time.rename(columns={"tiempo_inferencia": "tiempo_promedio_s"}, inplace=True)

# Tamaños
df_size = (
    df.groupby(["modelo"])["tamano_MB"]
    .first()
    .reset_index()
)

# Merge final
benchmark = (
    df_acc
    .merge(df_time, on=["modelo", "dataset"], how="right")
    .merge(df_size, on="modelo", how="left")
)

benchmark.to_csv("benchmark_fewshot.csv", index=False)

print("\nBenchmark guardado en benchmark_fewshot.csv")
print(benchmark)


Cargando datasets...
✔ Dataset médico: 0
✔ Dataset AG News: 200

Cargando modelo: google/flan-t5-small


Device set to use cuda:0



=== Dataset: medical ===


0it [00:00, ?it/s]



=== Dataset: news ===


...


=== Dataset: medical ===


0it [00:00, ?it/s]



=== Dataset: news ===


100%|██████████| 200/200 [01:35<00:00,  2.09it/s]



✔ Resultados guardados en few_shot_results_full.csv

Generando benchmark...

✔ Benchmark guardado en benchmark_fewshot.csv
                    modelo dataset  accuracy  tiempo_promedio_s      tamano_MB
0  EleutherAI/gpt-neo-1.3B    news     0.610           0.478303  NO DISPONIBLE
1  EleutherAI/gpt-neo-125M    news     0.610           0.223640  NO DISPONIBLE
2     google/flan-t5-large    news     0.610           0.314289  NO DISPONIBLE
3     google/flan-t5-small    news     0.645           0.088496  NO DISPONIBLE


# **Zero Shot**

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm
import pandas as pd
import gc
import torch


#  Cargar datasets en streaming (200 muestras cada uno)

print("Cargando dataset médico (streaming)…")
texts_medical = []
ds_med = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train", streaming=True)
for ex in tqdm(ds_med):
    if "input" in ex and "output" in ex:
        texts_medical.append(ex["input"] + " " + ex["output"])
    if len(texts_medical) >= 200:
        break
print("Dataset médico cargado:", len(texts_medical))

print("Cargando dataset PDF (streaming)…")
texts_pdfs = []
ds_pdfs = load_dataset("HuggingFaceFW/finepdfs", split="train", streaming=True)
for ex in tqdm(ds_pdfs):
    if "text" in ex and isinstance(ex["text"], str):
        texts_pdfs.append(ex["text"])
    if len(texts_pdfs) >= 200:
        break
print("Dataset PDF cargado:", len(texts_pdfs))

datasets_zero = [
    ("medical", texts_medical),
    ("pdf", texts_pdfs)
]


#  Modelos zero-shot

zero_models = [
    ("facebook/bart-large-mnli", "BART (large)"),
    ("valhalla/distilbart-mnli-12-1", "DistilBART (small)"),
    ("joeddav/xlm-roberta-large-xnli", "XLM-R (large)"),
    ("typeform/distilroberta-base", "DistilRoBERTa (small)")
]


#  Definir etiquetas

labels = ["positive", "negative"]


#  Loop principal

results = []

for model_name, model_size in zero_models:
    print(f"\nCargando modelo: {model_name}")
    try:
        classifier = pipeline("zero-shot-classification", model=model_name, device_map="auto")

        for dataset_name, textos in datasets_zero:
            print(f"\nDataset: {dataset_name} ")
            for texto in tqdm(textos):
                try:
                    output = classifier(texto, labels)
                    predicted = output["labels"][0]
                    score = output["scores"][0]

                    results.append({
                        "dataset": dataset_name,
                        "modelo": model_name,
                        "tamaño": model_size,
                        "texto": texto[:100] + "...",
                        "prediccion": predicted,
                        "confianza": score
                    })
                except Exception as e:
                    print(f"Error procesando texto: {e}")

    except Exception as e:
        print(f"Error cargando modelo {model_name}: {e}")


# Guardar resultados

df_results = pd.DataFrame(results)
df_results.to_csv("zero_shot_results_200.csv", index=False)
print("\nResultados guardados en zero_shot_results_200.csv")

Cargando dataset médico (streaming)…


19704it [00:04, 4805.56it/s]


Dataset médico cargado: 0
Cargando dataset PDF (streaming)…


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

...

Error procesando texto: The expanded size of the tensor (1623) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 1623].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (664) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 664].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (743) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 743].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (777) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 777].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (1601) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 1601].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (5842) must match the existing size (514) at non-singleton d


 95%|█████████▌| 190/200 [00:08<00:00, 51.86it/s]

Error procesando texto: The expanded size of the tensor (4343) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 4343].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (31548) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 31548].  Tensor sizes: [1, 514]



100%|██████████| 200/200 [00:09<00:00, 20.53it/s]

Error procesando texto: The expanded size of the tensor (528184) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 528184].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (549) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 549].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (837) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 837].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (2250) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 2250].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (3884) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 3884].  Tensor sizes: [1, 514]
Error procesando texto: The expanded size of the tensor (622) must match the existing size (514) at non-single

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
from datasets import load_dataset
from transformers import pipeline
from huggingface_hub import model_info
from tqdm import tqdm
import pandas as pd
import gc
import torch
import time


# ============================================================
# 1. Cargar datasets (streaming + AG News para accuracy real)
# ============================================================

print("Cargando dataset médico (streaming)…")
texts_medical = []
ds_med = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en",
                      split="train", streaming=True)

for ex in tqdm(ds_med):
    if "input" in ex and "output" in ex:
        texts_medical.append(ex["input"] + " " + ex["output"])
    if len(texts_medical) >= 200:
        break

print("Dataset médico cargado:", len(texts_medical))


print("Cargando dataset PDF (streaming)…")
texts_pdfs = []
ds_pdfs = load_dataset("HuggingFaceFW/finepdfs", split="train", streaming=True)

for ex in tqdm(ds_pdfs):
    if "text" in ex and isinstance(ex["text"], str):
        texts_pdfs.append(ex["text"])
    if len(texts_pdfs) >= 200:
        break

print("Dataset PDF cargado:", len(texts_pdfs))


print("Cargando AG News (200 muestras) para accuracy real…")
ds_news = load_dataset("ag_news", split="train[:200]")
texts_news = [ex["text"] for ex in ds_news]
labels_news = [ex["label"] for ex in ds_news]


# ============================================================
# 2. Crear etiquetas reales (positivas/negativas verificables)
# ============================================================

# Conversión estándar:
# 0 = World → negativo
# 1 = Sports → positivo
# 2 = Business → negativo
# 3 = Sci/Tech → positivo
label_map = {0: "negative", 1: "positive", 2: "negative", 3: "positive"}
labels_news_bin = [label_map[x] for x in labels_news]

# Dataset médico y pdf → sin etiquetas reales
labels_medical = ["unknown"] * len(texts_medical)
labels_pdfs = ["unknown"] * len(texts_pdfs)


datasets_zero = [
    ("medical", texts_medical, labels_medical),
    ("pdf", texts_pdfs, labels_pdfs),
    ("agnews", texts_news, labels_news_bin)  # este sí tiene accuracy real
]


# ============================================================
# 3. Modelos zero-shot a evaluar
# ============================================================

zero_models = [
    ("facebook/bart-large-mnli", "BART (large)"),
    ("valhalla/distilbart-mnli-12-1", "DistilBART (small)"),
    ("joeddav/xlm-roberta-large-xnli", "XLM-R (large)"),
    ("typeform/distilroberta-base", "DistilRoBERTa (small)")
]


# Etiquetas a clasificar
labels = ["positive", "negative"]


# ============================================================
# 4. Loop principal con tiempo + predicción + accuracy
# ============================================================

results = []

for model_name, model_size in zero_models:
    print(f"\nCargando modelo: {model_name}")

    try:
        classifier = pipeline("zero-shot-classification",
                              model=model_name,
                              device_map="auto")

        # Tamaño del modelo (MB)
        try:
            info = model_info(model_name)
            size_mb = round(info.siblings[0].size / (1024 * 1024), 2)
        except:
            size_mb = "NO DISPONIBLE"

        for dataset_name, textos, etiquetas_reales in datasets_zero:
            print(f"\nDataset: {dataset_name}")

            for texto, etiqueta_real in tqdm(list(zip(textos, etiquetas_reales))):
                try:
                    t0 = time.time()
                    output = classifier(texto, labels)
                    t1 = time.time()

                    pred = output["labels"][0]
                    score = output["scores"][0]
                    tiempo_inf = t1 - t0

                    acierto = (pred == etiqueta_real) if etiqueta_real != "unknown" else None

                    results.append({
                        "modelo": model_name,
                        "tamaño": model_size,
                        "dataset": dataset_name,
                        "texto": texto[:100] + "...",
                        "prediccion": pred,
                        "label_real": etiqueta_real,
                        "acierto": acierto,
                        "confianza": score,
                        "tiempo_inferencia": tiempo_inf,
                        "tamano_MB": size_mb
                    })

                except Exception as e:
                    print("Error procesando texto:", e)

    except Exception as e:
        print("Error cargando modelo:", e)

    del classifier
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 5. Guardar resultados completos
# ============================================================

df = pd.DataFrame(results)
df.to_csv("zero_shot_results_full.csv", index=False)
print("\n✔ Resultados guardados en zero_shot_results_full.csv")


# ============================================================
# 6. Generar Benchmark final
# ============================================================

print("Generando benchmark...")

# Accuracy (solo AG News tiene etiquetas)
df_acc = (
    df[df["acierto"].notna()]
    .groupby(["modelo", "dataset"])["acierto"]
    .mean()
    .reset_index()
)
df_acc.rename(columns={"acierto": "accuracy"}, inplace=True)

# Tiempos de inferencia
df_time = (
    df.groupby(["modelo", "dataset"])["tiempo_inferencia"]
    .mean()
    .reset_index()
)
df_time.rename(columns={"tiempo_inferencia": "tiempo_promedio_s"}, inplace=True)

# Tamaño
df_size = (
    df.groupby(["modelo"])["tamano_MB"]
    .first()
    .reset_index()
)

# MERGE FINAL
benchmark = (
    df_acc
    .merge(df_time, on=["modelo", "dataset"], how="right")
    .merge(df_size, on="modelo", how="left")
)

benchmark.to_csv("benchmark_zero_shot.csv", index=False)

print("\nBenchmark guardado en benchmark_zero_shot.csv")
print(benchmark)


# **Generacion de Texto**

In [ ]:
# IMPORTS Y TABLA DE RESULTADOS
import torch, time, math, pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

bench = []

def log_bench(**kwargs):
    bench.append(kwargs)

def bench_df():
    return pd.DataFrame(bench)

# CARGAR DATASETS Y CREAR TEXTOS
print("Cargando datasets...")

ds_medical = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en", split="train", streaming=True
)

ds_pdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    split="train", streaming=True
)

texts_medical, texts_pdfs = [], []

# 500 muestras de dataset médico
for ex in ds_medical:
    if "input" in ex and "output" in ex:
        texts_medical.append(ex["input"] + " " + ex["output"])
    if len(texts_medical) >= 500:
        break

# 500 muestras de PDF
for ex in ds_pdfs:
    if "text" in ex and isinstance(ex["text"], str):
        texts_pdfs.append(ex["text"])
    if len(texts_pdfs) >= 500:
        break

texts = texts_medical + texts_pdfs
print("Total textos:", len(texts))

# FUNCIÓN PARA PERPLEXITY MANUAL
def compute_ppl_manual(model, tokenizer, texts, max_len=1024):
    model.eval()
    losses = []

    for t in texts:
        enc = tokenizer(
            t, return_tensors="pt", truncation=True, max_length=max_len
        )
        input_ids = enc.input_ids.to(model.device)

        with torch.no_grad():
            outputs = model(input_ids, labels=input_ids)
            losses.append(outputs.loss.item())

    avg_loss = sum(losses) / len(losses)
    return math.exp(avg_loss)

# FUNCIÓN FINAL DE EVALUACIÓN
def run_generation(mid, size, texts, n=1000):
    print(f"\nEvaluando modelo: {mid} ({size}) ")

    tokenizer = AutoTokenizer.from_pretrained(mid)

    # GPU si existe, si no CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device usado:", device)

    model = AutoModelForCausalLM.from_pretrained(
        mid,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

    # Truncar cada texto a 1024 tokens
    clean = []
    for t in texts[:n]:
        ids = tokenizer.encode(t, add_special_tokens=False)
        clean.append(tokenizer.decode(ids[:1024]))

    print("Textos usados:", len(clean))

    # Medir inferencia
    gen_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=0 if device == "cuda" else -1,
        max_new_tokens=50,
        batch_size=1
    )

    t0 = time.time()
    _ = gen_pipe(clean[0])
    infer_time = time.time() - t0

    # Perplexity manual
    ppl = compute_ppl_manual(model, tokenizer, clean)

    # Registrar resultados
    params = sum(p.numel() for p in model.parameters())
    log_bench(
        task="text_generation",
        dataset="medical-o1 + finepdfs",
        n_samples=len(clean),
        model_id=mid,
        size_hint=size,
        metric_name="perplexity",
        metric_value=ppl,
        inference_time_s=infer_time,
        params_count=params,
        notes="Perplexity manual, trunc=1024"
    )

    print(f"→ {mid} | PPL: {ppl:.2f} | tiempo: {infer_time:.2f}s")

# MODELOS
models = [
    ("openai-community/gpt2", "small"),
    ("gpt2-medium", "medium"),
    ("EleutherAI/gpt-neo-125M", "small"),
    ("EleutherAI/gpt-neo-1.3B", "large")
]

# EJECUTAR BENCHMARK
for mid, sz in models:
    run_generation(mid, sz, texts, n=1000)

df = bench_df()
df
df.to_csv("benchmark_text_generation_1000.csv", index=False)

print("\nBenchmark guardado en benchmark_text_generation_1000.csv")

Cargando datasets...


Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

Total textos: 500

=== Evaluando modelo: openai-community/gpt2 (small) ===


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

Textos usados: 500
→ gpt2-medium | PPL: 12.38 | tiempo: 0.90s

=== Evaluando modelo: EleutherAI/gpt-neo-125M (small) ===
Device usado: cuda


Token indices sequence length is longer than the specified maximum sequence length for this model (5375 > 2048). Running this sequence through the model will result in indexing errors
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Textos usados: 500
→ EleutherAI/gpt-neo-125M | PPL: 13.44 | tiempo: 0.90s

=== Evaluando modelo: EleutherAI/gpt-neo-1.3B (large) ===
Device usado: cuda


Token indices sequence length is longer than the specified maximum sequence length for this model (5375 > 2048). Running this sequence through the model will result in indexing errors
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Textos usados: 500
→ EleutherAI/gpt-neo-1.3B | PPL: 8.62 | tiempo: 1.41s

Benchmark guardado en benchmark_text_generation_1000.csv


In [ ]:
df = bench_df()
df

,task,dataset,n_samples,model_id,size_hint,metric_name,metric_value,inference_time_s,params_count,notes
0,text_generation,medical-o1 + finepdfs,500,openai-community/gpt2,small,perplexity,16.034339,1.982057,124439808,"Perplexity manual, trunc=1024"
1,text_generation,medical-o1 + finepdfs,500,gpt2-medium,medium,perplexity,12.376363,0.899617,354823168,"Perplexity manual, trunc=1024"
2,text_generation,medical-o1 + finepdfs,500,EleutherAI/gpt-neo-125M,small,perplexity,13.439127,0.897689,125198592,"Perplexity manual, trunc=1024"
3,text_generation,medical-o1 + finepdfs,500,EleutherAI/gpt-neo-1.3B,large,perplexity,8.616666,1.411789,1315575808,"Perplexity manual, trunc=1024"


# **Prueba de modelos de generacion de texto**

In [ ]:
from ipywidgets import interact, Dropdown, Textarea
from transformers import pipeline

modelos = [
    "openai-community/gpt2",
    "gpt2-medium",
    "EleutherAI/gpt-neo-125M",
    "EleutherAI/gpt-neo-1.3B"
]

def probar_modelo(modelo, prompt):
    gen = pipeline("text-generation", model=modelo, max_new_tokens=1000)
    print(gen(prompt)[0]["generated_text"])

interact(
    probar_modelo,
    modelo=Dropdown(options=modelos),
    prompt=Textarea(value="Write something:")
)


interactive(children=(Dropdown(description='modelo', options=('openai-community/gpt2', 'gpt2-medium', 'Eleuthe…

<function __main__.probar_modelo(modelo, prompt)>